# Vietnam Egg Price Intelligence — Preprocessing v2

This notebook keeps the MASTER file unchanged and creates dashboard-ready CSV outputs for Streamlit and Looker.

Key additions in this version:
- mixed-date parsing and field type normalization;
- retail egg-count and price outlier checks;
- pack-price / per-egg consistency checks;
- client-facing **Caged / Cage-Free / Free-Range** housing classification;
- FeedIn treated as **Caged (provisional)** pending confirmation;
- AGROINFO historical market data treated as **Caged (provisional)** for current analytics;
- WinMart treated as the **Caged retail baseline** unless explicit non-caged wording appears;
- Unknown housing remains in the database but is excluded from housing-comparison charts.

In [65]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)

PROJECT_DIR = Path.cwd()
MASTER_FILE = PROJECT_DIR / "MASTER_EGG_PRICE_DATA.csv"

if not MASTER_FILE.is_file():
    raise FileNotFoundError(
        f"MASTER_EGG_PRICE_DATA.csv was not found in the project folder: {PROJECT_DIR}"
    )

OUTPUT_DIR = PROJECT_DIR / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_REASONABLE_PRICE_PER_EGG = 500
MAX_REASONABLE_PRICE_PER_EGG = 20_000
SOFT_RETAIL_EGG_COUNT = 60
HARD_RETAIL_EGG_COUNT = 180
FEEDIN_UNIT_CONFIRMED = False

print("Project folder:", PROJECT_DIR)
print("MASTER file:", MASTER_FILE)
print("Output folder:", OUTPUT_DIR)

Project folder: C:\Users\admin\PycharmProjects\egg_price_project_vietnam
MASTER file: C:\Users\admin\PycharmProjects\egg_price_project_vietnam\MASTER_EGG_PRICE_DATA.csv
Output folder: C:\Users\admin\PycharmProjects\egg_price_project_vietnam\processed_data


## Helpers and housing rules

In [66]:
def clean_text(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ''
    return ' '.join(str(value).split())


def normalize_text(value):
    text = clean_text(value)
    if not text:
        return ''
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^a-zA-Z0-9]+', ' ', text)
    return ' '.join(text.lower().split())


def clean_identifier(series):
    s = series.astype('string').str.strip()
    s = s.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, 'none': pd.NA})
    return s.str.replace(r'\.0$', '', regex=True)


FREE_RANGE_TERMS = [normalize_text(x) for x in (
    'free-range', 'free range', 'thả vườn', 'gà thả vườn', 'trứng gà thả vườn',
    'thả rông', 'outdoor access', 'pasture raised', 'pasture-raised'
)]
CAGE_FREE_TERMS = [normalize_text(x) for x in (
    'cage-free', 'cage free', 'cagefree', 'không lồng', 'không nhốt lồng',
    'không nuôi lồng', 'gà không lồng', 'gà nuôi không lồng', 'trứng gà nhân đạo',
    'trứng nhân đạo', 'Certified Humane', 'HFAC', 'Kê Phi'
)]


def contains_any(text, terms):
    return any(term and term in text for term in terms)


def classify_housing(row):
    source = normalize_text(row.get('Source'))
    raw = normalize_text(row.get('Production System'))
    # Use only genuine source/product wording for explicit overrides. Do not use
    # the old derived Egg Type fields, because older WinMart logic mislabeled
    # ga ta / ga que as free-range.
    evidence = normalize_text(' '.join([
        clean_text(row.get('Product Name')),
        clean_text(row.get('Notes')),
    ]))

    # Client-approved provisional market-data rules while source confirmation is pending.
    if source == 'feedin':
        return pd.Series(['Caged', 'Provisional', 'FeedIn treated as Caged pending source confirmation'])

    if source == 'agroinfo':
        return pd.Series(['Caged', 'Provisional', 'AGROINFO historical market data treated as Caged for current analytics'])

    # WinMart is the project's primary caged retail baseline. Explicit non-caged
    # product wording would override the default if such a SKU appears later.
    if source == 'winmart':
        if contains_any(evidence, FREE_RANGE_TERMS):
            return pd.Series(['Free-Range', 'Explicit', 'Explicit free-range wording in product/source text'])
        if contains_any(evidence, CAGE_FREE_TERMS):
            return pd.Series(['Cage-Free', 'Explicit', 'Explicit cage-free wording in product/source text'])
        return pd.Series(['Caged', 'Project Rule', 'WinMart caged retail baseline'])

    # For MM/eMart/Farmers and other sources, explicit wording is strongest.
    if contains_any(evidence, FREE_RANGE_TERMS):
        return pd.Series(['Free-Range', 'Explicit', 'Explicit free-range wording in product/source text'])
    if contains_any(evidence, CAGE_FREE_TERMS):
        return pd.Series(['Cage-Free', 'Explicit', 'Explicit cage-free wording in product/source text'])

    raw_map = {
        'caged': 'Caged',
        'cage free': 'Cage-Free',
        'cagefree': 'Cage-Free',
        'free range': 'Free-Range',
    }
    if raw in raw_map:
        return pd.Series([raw_map[raw], 'Source Provided', 'Existing Production System label retained'])

    return pd.Series(['Unknown', 'Unknown', 'No reliable housing classification available'])

## Load and normalize fields

In [67]:
# ------------------------------------------------------------------
# LOAD / BASIC CLEANUP

## Housing classification

In [68]:
# ------------------------------------------------------------------
df_raw = pd.read_csv(MASTER_FILE, encoding='utf-8-sig', low_memory=False)
df = df_raw.copy()

missing_text_values = {'', 'nan', 'none', 'null', 'n/a', 'na'}
for col in df.columns:
    if df[col].dtype == 'object':
        s = df[col].astype('string').str.strip()
        df[col] = s.mask(s.str.lower().isin(missing_text_values), pd.NA)

# Keep identifiers as strings instead of values such as 10606717.0.
for col in ['Record ID', 'Item No', 'Store Code', 'Store Group Code']:
    if col in df.columns:
        df[col] = clean_identifier(df[col])

numeric_columns = [
    'Egg Count', 'Price Raw', 'Buying Price VND', 'Selling Price VND',
    'Pack Price VND', 'Price Per Egg VND', 'Quantity Sold', 'Stock Quantity',
    'Feed Cost VND',
]
for col in numeric_columns:
    if col in df.columns:
        if df[col].dtype == 'object' or str(df[col].dtype).startswith('string'):
            df[col] = df[col].astype('string').str.replace(',', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Mixed legacy + ISO dates are intentional.
df['Date Clean'] = pd.to_datetime(df['Date'], errors='coerce', format='mixed').dt.normalize()
df['Scrape Timestamp Clean'] = pd.to_datetime(
    df['Scrape Timestamp'], errors='coerce', format='mixed', utc=True
).dt.tz_convert('Asia/Ho_Chi_Minh').dt.tz_localize(None)

df['Price Level'] = df['Market Level'].astype('string').str.strip().str.title()
df['Region Normalized'] = df['Region'].astype('string').str.strip().str.title()
df['Region Normalized'] = df['Region Normalized'].replace({'Unknown': pd.NA})
df['Province Normalized'] = df['Province'].astype('string').str.strip()
df['City Normalized'] = df['City'].astype('string').str.strip()

# Data origin is useful for dashboard transparency.
def data_origin(row):
    st = normalize_text(row.get('Source Type'))
    if st in {'historical database', 'historical backfill', 'compiled database'}:
        return 'Historical / Compiled'
    if st in {'retail', 'market'}:
        return 'Automated / Current'
    return 'Other'

df['Data Origin'] = df.apply(data_origin, axis=1)

## Price and egg-count validation

In [69]:
# ------------------------------------------------------------------
# HOUSING CLASSIFICATION

## Duplicate and quality checks

In [70]:
# ------------------------------------------------------------------
df[['Housing System', 'Housing Label Status', 'Housing Label Basis']] = df.apply(
    classify_housing, axis=1
)
df['Housing Dashboard Eligible'] = np.where(
    df['Housing System'].isin(['Caged', 'Cage-Free', 'Free-Range']), 'Yes', 'No'
)

## Analytics eligibility

In [71]:
# ------------------------------------------------------------------
# PRICE / EGG COUNT VALIDATION

## Summaries

In [72]:
# ------------------------------------------------------------------
df['Egg Count Clean'] = df['Egg Count'].copy()
df['Price Per Egg VND Clean'] = df['Price Per Egg VND'].copy()

retail_mask = df['Price Level'].eq('Retail')
hard_pack_outlier = retail_mask & (
    df['Egg Count Clean'].isna()
    | df['Egg Count Clean'].le(0)
    | df['Egg Count Clean'].gt(HARD_RETAIL_EGG_COUNT)
)
# Missing pack count is only a hard problem when a pack price exists; historical
# per-egg retail observations legitimately use Egg Count = 1 and no pack price.
hard_pack_outlier = hard_pack_outlier & (
    df['Pack Price VND'].notna() | df['Egg Count Clean'].notna()
)

# Large packs such as MM Mega Market cartons can be legitimate. Keep 61-180 egg
# packs but flag them for review; only >180 is treated as a hard outlier.
large_pack_review = retail_mask & df['Egg Count Clean'].gt(SOFT_RETAIL_EGG_COUNT) & df['Egg Count Clean'].le(HARD_RETAIL_EGG_COUNT)

df.loc[hard_pack_outlier, ['Egg Count Clean', 'Price Per Egg VND Clean']] = np.nan

# For retail pack observations, recompute consistently from pack price/count.
can_calculate_retail = (
    retail_mask
    & df['Pack Price VND'].notna()
    & df['Egg Count Clean'].notna()
    & df['Egg Count Clean'].gt(0)
)
calculated_retail_price = df['Pack Price VND'] / df['Egg Count Clean']
pack_price_mismatch = (
    can_calculate_retail
    & df['Price Per Egg VND'].notna()
    & (calculated_retail_price - df['Price Per Egg VND']).abs().gt(2)
)
df.loc[can_calculate_retail, 'Price Per Egg VND Clean'] = calculated_retail_price[can_calculate_retail]

suspect_price = df['Price Per Egg VND Clean'].notna() & ~df['Price Per Egg VND Clean'].between(
    MIN_REASONABLE_PRICE_PER_EGG, MAX_REASONABLE_PRICE_PER_EGG
)
df.loc[suspect_price, 'Price Per Egg VND Clean'] = np.nan

# Use nullable integer for clean pack counts when possible.
df['Egg Count Clean'] = df['Egg Count Clean'].round().astype('Int64')

## Export and hard checks

In [73]:
# ------------------------------------------------------------------
# DUPLICATE / QUALITY CHECKS

In [74]:
# ------------------------------------------------------------------
duplicate_key = [
    'Date Clean', 'Source', 'Price Level', 'Province Normalized', 'City Normalized',
    'Location', 'Store Name', 'Egg Type Normalized', 'Brand', 'Product Name',
    'Pack Size', 'Price Per Egg VND Clean',
]
duplicate_key = [c for c in duplicate_key if c in df.columns]
df['Duplicate Flag'] = np.where(df.duplicated(subset=duplicate_key, keep=False), 'REVIEW', 'OK')


def base_quality_status(value):
    text = clean_text(value).upper()
    if text.startswith('ERROR'):
        return 'ERROR'
    if text.startswith('REVIEW'):
        return 'REVIEW'
    if text.startswith('OK'):
        return 'OK'
    return 'REVIEW'


df['Quality Status'] = df['Quality Flag'].apply(base_quality_status)
df['Preprocessing Issues'] = ''


def add_issue(mask, issue, status='REVIEW'):
    mask = pd.Series(mask, index=df.index).fillna(False)
    current = df.loc[mask, 'Preprocessing Issues'].astype('string')
    df.loc[mask, 'Preprocessing Issues'] = np.where(
        current.eq('') | current.isna(), issue, current + '; ' + issue
    )
    if status == 'ERROR':
        df.loc[mask, 'Quality Status'] = 'ERROR'
    elif status == 'REVIEW':
        reviewable = mask & ~df['Quality Status'].eq('ERROR')
        df.loc[reviewable, 'Quality Status'] = 'REVIEW'


add_issue(df['Date Clean'].isna(), 'INVALID_OR_MISSING_DATE', 'ERROR')
add_issue(df['Price Per Egg VND'].isna(), 'MISSING_PRICE_PER_EGG', 'REVIEW')
add_issue(df['Price Level'].isna(), 'MISSING_PRICE_LEVEL', 'REVIEW')
add_issue(hard_pack_outlier, 'SUSPECT_RETAIL_PACK_SIZE', 'ERROR')
add_issue(large_pack_review, 'LARGE_RETAIL_PACK_REVIEW', 'REVIEW')
add_issue(suspect_price, 'PRICE_OUTSIDE_QC_RANGE', 'REVIEW')
add_issue(pack_price_mismatch, 'PACK_PRICE_PER_EGG_MISMATCH', 'REVIEW')
add_issue(df['Duplicate Flag'].eq('REVIEW'), 'POSSIBLE_DUPLICATE', 'REVIEW')

missing_location = (
    df['Province Normalized'].isna()
    & df['City Normalized'].isna()
    & df['Location'].isna()
    & df['Store Name'].isna()
)
add_issue(missing_location, 'MISSING_LOCATION', 'REVIEW')

# FeedIn's published unit is still being confirmed. Keep the price usable but transparent.
if not FEEDIN_UNIT_CONFIRMED:
    feedin_mask = df['Source'].astype('string').str.casefold().eq('feedin')
    add_issue(feedin_mask, 'FEEDIN_UNIT_NEEDS_CONFIRMATION', 'REVIEW')

add_issue(df['Housing System'].eq('Unknown'), 'HOUSING_SYSTEM_UNCLASSIFIED', 'REVIEW')
df['Preprocessing Issues'] = df['Preprocessing Issues'].replace('', pd.NA)

In [75]:
# ------------------------------------------------------------------
# ANALYTICS ELIGIBILITY

In [76]:
# ------------------------------------------------------------------
df_processed = df.sort_values(['Date Clean', 'Source', 'Price Level'], na_position='last').reset_index(drop=True)

basic_eligible = (
    ~df_processed['Quality Status'].eq('ERROR')
    & df_processed['Date Clean'].notna()
    & df_processed['Price Per Egg VND Clean'].notna()
)

duplicate_extra = pd.Series(False, index=df_processed.index)
if duplicate_key:
    duplicate_extra.loc[basic_eligible] = (
        df_processed.loc[basic_eligible].duplicated(subset=duplicate_key, keep='first').to_numpy()
    )

df_processed['Analytics Eligible'] = 'Yes'
df_processed['Analytics Exclusion Reason'] = pd.NA
error_mask = df_processed['Quality Status'].eq('ERROR')
invalid_date_mask = df_processed['Date Clean'].isna()
missing_clean_price_mask = df_processed['Price Per Egg VND Clean'].isna()

df_processed.loc[error_mask, ['Analytics Eligible', 'Analytics Exclusion Reason']] = ['No', 'QUALITY_ERROR']
df_processed.loc[~error_mask & invalid_date_mask, ['Analytics Eligible', 'Analytics Exclusion Reason']] = ['No', 'INVALID_OR_MISSING_DATE']
df_processed.loc[~error_mask & ~invalid_date_mask & missing_clean_price_mask, ['Analytics Eligible', 'Analytics Exclusion Reason']] = ['No', 'NO_VALID_CLEAN_PRICE']
df_processed.loc[duplicate_extra, ['Analytics Eligible', 'Analytics Exclusion Reason']] = ['No', 'DUPLICATE_COPY']

# Calendar fields.
df_processed['Year'] = df_processed['Date Clean'].dt.year.astype('Int64')
df_processed['Month Number'] = df_processed['Date Clean'].dt.month.astype('Int64')
df_processed['Month'] = df_processed['Date Clean'].dt.strftime('%b')
df_processed['Year Month'] = df_processed['Date Clean'].dt.to_period('M').astype('string')
df_processed['Quarter'] = 'Q' + df_processed['Date Clean'].dt.quarter.astype('Int64').astype('string')
df_processed['Week'] = df_processed['Date Clean'].dt.isocalendar().week.astype('Int64')
df_processed['Day of Week'] = df_processed['Date Clean'].dt.day_name()
df_processed['Analytics Price Available'] = np.where(df_processed['Price Per Egg VND Clean'].notna(), 'Yes', 'No')

# Standardized output formats for CSV, BigQuery, Streamlit, and Looker Studio.
# Observation dates use ISO 8601: YYYY-MM-DD.
date_iso = pd.to_datetime(
    df_processed['Date Clean'],
    errors='coerce',
).dt.strftime('%Y-%m-%d')

# Keep Date and Date Clean consistent in the ANALYTICS output.
# MASTER_EGG_PRICE_DATA.csv itself is never modified.
df_processed['Date'] = date_iso
df_processed['Date Clean'] = date_iso

# Scrape timestamps keep their time component.
timestamp_iso = pd.to_datetime(
    df_processed['Scrape Timestamp Clean'],
    errors='coerce',
).dt.strftime('%Y-%m-%d %H:%M:%S')

df_processed['Scrape Timestamp'] = timestamp_iso
df_processed['Scrape Timestamp Clean'] = timestamp_iso

df_analytics = df_processed[df_processed['Analytics Eligible'].eq('Yes')].copy().reset_index(drop=True)
df_quality = df_processed[df_processed['Quality Status'].isin(['REVIEW', 'ERROR'])].copy()

In [77]:
# ------------------------------------------------------------------
# SUMMARIES

In [78]:
# ------------------------------------------------------------------
housing_summary = (
    df_analytics.groupby(['Housing System', 'Price Level'], dropna=False)
    .agg(Rows=('Record ID', 'size'), Sources=('Source', 'nunique'), First_Date=('Date Clean', 'min'), Last_Date=('Date Clean', 'max'), Average_Price_VND=('Price Per Egg VND Clean', 'mean'))
    .reset_index()
)

source_housing_summary = (
    df_analytics.groupby(['Source', 'Price Level', 'Housing System', 'Housing Label Status'], dropna=False)
    .agg(Rows=('Record ID', 'size'), Average_Price_VND=('Price Per Egg VND Clean', 'mean'))
    .reset_index()
)

issue_summary = (
    df_processed.assign(Issue=df_processed['Preprocessing Issues'].fillna('NO_ISSUE'))
    .groupby(['Quality Status', 'Issue'], dropna=False)
    .size().reset_index(name='Rows')
    .sort_values('Rows', ascending=False)
)

qa_rows = [
    ('Raw MASTER rows', len(df_raw)),
    ('Clean analytics rows', len(df_analytics)),
    ('Rows excluded', len(df_processed)-len(df_analytics)),
    ('ERROR rows in analytics', int(df_analytics['Quality Status'].eq('ERROR').sum())),
    ('Missing clean prices in analytics', int(df_analytics['Price Per Egg VND Clean'].isna().sum())),
    ('Hard retail pack outliers in analytics', int(((df_analytics['Price Level']=='Retail') & (pd.to_numeric(df_analytics['Egg Count Clean'], errors='coerce')>HARD_RETAIL_EGG_COUNT)).sum())),
    ('Raw large retail packs >60 and <=180 eggs', int((retail_mask & df['Egg Count'].gt(SOFT_RETAIL_EGG_COUNT) & df['Egg Count'].le(HARD_RETAIL_EGG_COUNT)).sum())),
    ('Raw hard retail pack outliers >180 eggs', int((retail_mask & df['Egg Count'].gt(HARD_RETAIL_EGG_COUNT)).sum())),
    ('Raw price outliers <500 or >20000 VND/egg', int((~df['Price Per Egg VND'].between(MIN_REASONABLE_PRICE_PER_EGG, MAX_REASONABLE_PRICE_PER_EGG)).sum())),
    ('Raw missing Production System', int(df['Production System'].isna().sum())),
    ('Derived Housing Unknown', int(df_processed['Housing System'].eq('Unknown').sum())),
    ('Derived Caged rows', int(df_processed['Housing System'].eq('Caged').sum())),
    ('Derived Cage-Free rows', int(df_processed['Housing System'].eq('Cage-Free').sum())),
    ('Derived Free-Range rows', int(df_processed['Housing System'].eq('Free-Range').sum())),
]
qa_summary = pd.DataFrame(qa_rows, columns=['Check', 'Value'])

In [79]:
# ------------------------------------------------------------------
# EXPORT

In [80]:
# ------------------------------------------------------------------
analytics_path = OUTPUT_DIR / 'ANALYTICS_EGG_PRICE_DATA.csv'
quality_path = OUTPUT_DIR / 'DATA_QUALITY_ISSUES.csv'
housing_path = OUTPUT_DIR / 'HOUSING_COVERAGE_SUMMARY.csv'
source_housing_path = OUTPUT_DIR / 'SOURCE_HOUSING_COVERAGE.csv'
issue_path = OUTPUT_DIR / 'QUALITY_ISSUE_SUMMARY.csv'
qa_path = OUTPUT_DIR / 'DATASET_QA_SUMMARY.csv'

df_analytics.to_csv(analytics_path, index=False, encoding='utf-8-sig')
df_quality.to_csv(quality_path, index=False, encoding='utf-8-sig')
housing_summary.to_csv(housing_path, index=False, encoding='utf-8-sig')
source_housing_summary.to_csv(source_housing_path, index=False, encoding='utf-8-sig')
issue_summary.to_csv(issue_path, index=False, encoding='utf-8-sig')
qa_summary.to_csv(qa_path, index=False, encoding='utf-8-sig')

# Hard checks.
assert not df_analytics['Quality Status'].eq('ERROR').any()
assert df_analytics['Date Clean'].notna().all()
assert df_analytics['Price Per Egg VND Clean'].notna().all()
assert not ((df_analytics['Price Level'].eq('Retail')) & (pd.to_numeric(df_analytics['Egg Count Clean'], errors='coerce') > HARD_RETAIL_EGG_COUNT)).any()

print(qa_summary.to_string(index=False))
print('\nHousing summary:')
print(housing_summary.to_string(index=False))
print('\nOutputs:')
for x in [analytics_path, quality_path, housing_path, source_housing_path, issue_path, qa_path]:
    print(x)

                                    Check  Value
                          Raw MASTER rows   5830
                     Clean analytics rows   5786
                            Rows excluded     44
                  ERROR rows in analytics      0
        Missing clean prices in analytics      0
   Hard retail pack outliers in analytics      0
Raw large retail packs >60 and <=180 eggs      3
  Raw hard retail pack outliers >180 eggs     19
Raw price outliers <500 or >20000 VND/egg     19
            Raw missing Production System   4378
                  Derived Housing Unknown    313
                       Derived Caged rows   5423
                   Derived Cage-Free rows      5
                  Derived Free-Range rows     89

Housing summary:
Housing System Price Level  Rows  Sources First_Date  Last_Date  Average_Price_VND
     Cage-Free      Retail     5        2 2026-09-07 2026-09-11        3212.000000
         Caged      Market  4050       16 2010-01-05 2026-09-11        3122.68765